In [1]:
import os
import numpy as np
import csv

import matplotlib.pyplot as plt
import torch
from monai import transforms
from monai.data import DataLoader, Dataset, CacheDataset, SmartCacheDataset
from monai.utils import set_determinism
from monai.data.utils import pad_list_data_collate
from monai.inferers import SlidingWindowInferer
from torch.amp import autocast
from torch.nn import L1Loss
from tqdm import tqdm

from monai.inferers import LatentDiffusionInferer
from monai.losses import PatchAdversarialLoss, PerceptualLoss
from monai.networks.nets import AutoencoderKL, DiffusionModelUNet, PatchDiscriminator
from monai.networks.schedulers import DDPMScheduler

from torch.utils.tensorboard import SummaryWriter

#print_config()

## Patch-Based Latent Diffusion Model

In [2]:
#ROOT_DIR = "/home/fehrdelt/bettik/"
ROOT_DIR = "/bettik/PROJECTS/pr-gin5_aini/fehrdelt/"

In [3]:
# for reproducibility purposes set a seed
set_determinism(0)

## Using Final_ADC_Dataset

3D ADC volumes (225,225,225)

In [4]:
train_csv = os.path.join(ROOT_DIR, "AnoDiffExperiments/data_splits_lists/final_adc_dataset_small/train.csv")
train_images_path = []

with open(train_csv, mode='r') as file:
    reader = csv.reader(file)
    for line in tqdm(reader):
        #print(line)
        train_images_path.append(ROOT_DIR+line[0])

val_csv = os.path.join(ROOT_DIR, "AnoDiffExperiments/data_splits_lists/final_adc_dataset_small/val.csv")
val_images_path = []

with open(val_csv, mode='r') as file:
    reader = csv.reader(file)
    for line in tqdm(reader):

        val_images_path.append(ROOT_DIR+line[0])

test_reconstruction_csv = os.path.join(ROOT_DIR, "AnoDiffExperiments/data_splits_lists/final_adc_dataset_small/test.csv")
test_reconstruction_images_path = []

with open(test_reconstruction_csv, mode='r') as file:
    reader = csv.reader(file)
    for line in tqdm(reader):

        test_reconstruction_images_path.append(ROOT_DIR+line[0])


1636it [00:00, 75614.68it/s]
204it [00:00, 94776.03it/s]
205it [00:00, 3047.94it/s]


In [5]:
#train_datalist = sorted(train_images_path)
train_datalist = train_images_path

#val_datalist = sorted(val_images_path)
val_datalist = val_images_path

#val_datalist = sorted(val_images_path)
test_reconstruction_datalist = test_reconstruction_images_path

# Reduced batch size for memory efficiency
batch_size = 1  # Keep at 1 for 3D volumes with patches
num_workers = 2  # Reduced from 4 to save memory

print(f"Training on {len(train_datalist)} images")
print(f"Validation on {len(val_datalist)} images") 
print(f"Test reconstruction on {len(test_reconstruction_datalist)} images")
print(f"Batch size: {batch_size}, Workers: {num_workers}")

Training on 1636 images
Validation on 204 images
Test reconstruction on 205 images
Batch size: 1, Workers: 2


In [6]:
class SetBackgroundToZero(transforms.Transform):
    """
    Custom MONAI transform that zeros out voxels with the most frequent intensity value.
    
    Args:
        keys (str or list): Keys of the dictionary to apply the transform to.
        tolerance (int): Optional range around the mode value to also zero.
    """
    def __init__(self, tolerance: int = 0):
        super().__init__()
        self.tolerance = tolerance

    def __call__(self, data):
        
            
        is_tensor = isinstance(data, torch.Tensor)
        data_np = data.cpu().numpy() if is_tensor else data

        # Flatten and compute histogram
        flat = data_np.flatten()
        unique, counts = np.unique(flat, return_counts=True)
        mode_val = unique[np.argmax(counts)]

        # Apply tolerance if specified
        if self.tolerance > 0:
            mask = np.isin(data_np, range(mode_val - self.tolerance, mode_val + self.tolerance + 1))
        else:
            mask = data_np == mode_val

        # Zero out the background
        data_np[mask] = 0

        # Put back in original type
        data = torch.from_numpy(data_np) if is_tensor else data_np

        return data

In [7]:
IMAGE_SIZE = 128 # Reduced from 225 to save memory
PATCH_SIZE = 48  # Reduced from 64 to save memory (48^3 vs 64^3 = ~57% less memory)

print(f"Image size: {IMAGE_SIZE}^3, Patch size: {PATCH_SIZE}^3")
print(f"Memory reduction from patch size change: ~{(1 - (48**3)/(64**3))*100:.1f}%")

Image size: 128^3, Patch size: 48^3
Memory reduction from patch size change: ~57.8%


In [8]:
train_transforms = transforms.Compose(
    [
        transforms.LoadImage(),
        transforms.EnsureChannelFirst(),
        transforms.RandAffine(prob=0.2, rotate_range=(0.10, 0.10, 0.10), lazy=True),#+- 0.10 radians for each axis
        transforms.RandScaleCrop(roi_scale=0.9, max_roi_scale=1.1, random_size=True, lazy=True),
        transforms.ResizeWithPadOrCrop(spatial_size=(IMAGE_SIZE, IMAGE_SIZE, IMAGE_SIZE), lazy=True),
        transforms.ScaleIntensityRange(a_min=0.0, a_max=4000.0, b_min=0.0, b_max=1.0, clip=True),
        transforms.RandScaleIntensity(factors=0.15),
        transforms.RandFlip(prob=0.5, spatial_axis=0),
        SetBackgroundToZero(),
        #transforms.RandSpatialCrop(roi_size=(PATCH_SIZE, PATCH_SIZE, PATCH_SIZE)),  # Enable patch-based training
    ]
)

#train_ds = CacheDataset(data=train_datalist, transform=train_transforms)

train_ds = SmartCacheDataset(
    data=train_datalist,
    transform=train_transforms,
    cache_num=200,  # Reduced from 600 to use less RAM
    replace_rate=0.3,  # Increased to refresh cache more often
)

train_loader = DataLoader(
    #collate_fn=pad_list_data_collate: any tensors are centrally padded to match the shape of the biggest tensor in each dimension
    train_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, persistent_workers=True, collate_fn=pad_list_data_collate
)

Loading dataset: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 200/200 [00:51<00:00,  3.91it/s]


In [ ]:
# Visualize axial, coronal, and sagittal slices for the first 8 images in train_loader
n_samples = 8
for idx, batch in enumerate(train_loader):
    if idx >= n_samples:
        break
    img = batch[0, 0].detach().cpu().numpy()
    fig, axs = plt.subplots(1, 3, figsize=(6, 2))
    axs[0].imshow(img[:, :, img.shape[2] // 2], cmap="gray")
    axs[0].set_title(f"Axial {idx}")
    axs[0].axis("off")
    axs[1].imshow(img[:, img.shape[1] // 2, :], cmap="gray")
    axs[1].set_title("Coronal")
    axs[1].axis("off")
    axs[2].imshow(img[img.shape[0] // 2, :, :], cmap="gray")
    axs[2].set_title("Sagittal")
    axs[2].axis("off")
    plt.tight_layout()
    plt.show()

Sinon: pas utiliser GridPatchDataset mais juste mettre un random crop en transform train 

et à l'inférence prendre slidingwindowinferer

In [10]:
val_transforms = transforms.Compose(
    [
        transforms.LoadImage(),
        transforms.EnsureChannelFirst(),
        transforms.ScaleIntensityRange(a_min=0.0, a_max=4000.0, b_min=0.0, b_max=1.0, clip=True),
        transforms.Resize(spatial_size=(IMAGE_SIZE,IMAGE_SIZE,IMAGE_SIZE)),
        transforms.ResizeWithPadOrCrop(spatial_size=(IMAGE_SIZE,IMAGE_SIZE,IMAGE_SIZE), lazy=True),
        SetBackgroundToZero(),
        transforms.RandSpatialCrop(roi_size=(PATCH_SIZE, PATCH_SIZE, PATCH_SIZE)),  # Also use patches for validation
    ]
)
#val_ds = CacheDataset(data=val_datalist, transform=val_transforms)
val_ds = Dataset(data=val_datalist[:2], transform=val_transforms)  # Keep small validation set
val_loader = DataLoader(
    val_ds, batch_size=1, shuffle=False, num_workers=2, persistent_workers=False  # Reduced workers for validation
)

## Autoencoder KL

### Define Autoencoder KL network

In this section, we will define an autoencoder with KL-regularization for the LDM. The autoencoder's primary purpose is to transform input images into a latent representation that the diffusion model will subsequently learn. By doing so, we can decrease the computational resources required to train the diffusion component, making this approach suitable for learning high-resolution medical images.


In [11]:
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device="cuda:0"
print(f"Using {device}")

Using cuda:0


In [12]:
def print_memory_usage():
    """Print current GPU memory usage"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        max_allocated = torch.cuda.max_memory_allocated() / 1024**3
        print(f"GPU Memory - Allocated: {allocated:.2f} GB, Reserved: {reserved:.2f} GB, Max Allocated: {max_allocated:.2f} GB")
    else:
        print("CUDA not available")

# Monitor initial memory
print_memory_usage()

GPU Memory - Allocated: 0.00 GB, Reserved: 0.00 GB, Max Allocated: 0.00 GB


Autoencoder architecture: need to base on https://static-content.springer.com/esm/chp%3A10.1007%2F978-3-031-18576-2_12/MediaObjects/540283_1_En_12_MOESM1_ESM.pdf   

In [13]:
autoencoder = AutoencoderKL(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    channels=(32, 64, 64),  # Reduced from (32, 64, 64) (16, 32, 32)
    latent_channels=3,      # Reduced from 3 2
    num_res_blocks=1,
    norm_num_groups=16,      # Reduced from 16 8
    attention_levels=(False, False, True),
)
autoencoder.to(device)


discriminator = PatchDiscriminator(spatial_dims=3, num_layers_d=3, channels=32, in_channels=1, out_channels=1)  # Reduced layers and channels
#num_layers_d=2, channels=16, in_channels=1, out_channels=1
discriminator.to(device)

PatchDiscriminator(
  (initial_conv): Convolution(
    (conv): Conv3d(1, 32, kernel_size=(4, 4, 4), stride=(2, 2, 2), padding=(1, 1, 1))
    (adn): ADN(
      (D): Dropout(p=0.0, inplace=False)
      (A): LeakyReLU(negative_slope=0.2)
    )
  )
  (0): Convolution(
    (conv): Conv3d(32, 64, kernel_size=(4, 4, 4), stride=(2, 2, 2), padding=(1, 1, 1), bias=False)
    (adn): ADN(
      (N): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (D): Dropout(p=0.0, inplace=False)
      (A): LeakyReLU(negative_slope=0.2)
    )
  )
  (1): Convolution(
    (conv): Conv3d(64, 128, kernel_size=(4, 4, 4), stride=(2, 2, 2), padding=(1, 1, 1), bias=False)
    (adn): ADN(
      (N): BatchNorm3d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (D): Dropout(p=0.0, inplace=False)
      (A): LeakyReLU(negative_slope=0.2)
    )
  )
  (2): Convolution(
    (conv): Conv3d(128, 256, kernel_size=(4, 4, 4), stride=(1, 1, 1), padding=(1, 1, 1), bias=Fal

### Defining Losses

We will also specify the perceptual and adversarial losses, including the involved networks, and the optimizers to use during the training process.

In [14]:
l1_loss = L1Loss()
adv_loss = PatchAdversarialLoss(criterion="least_squares")
loss_perceptual = PerceptualLoss(spatial_dims=3, network_type="squeeze", is_fake_3d=True, fake_3d_ratio=0.2)
loss_perceptual.to(device)


def kl_loss(z_mu, z_sigma):
    klloss = 0.5 * torch.sum(z_mu.pow(2) + z_sigma.pow(2) - torch.log(z_sigma.pow(2)) - 1, dim=[1, 2, 3, 4])
    return torch.sum(klloss) / klloss.shape[0]


adv_weight = 0.01
perceptual_weight = 0.001
kl_weight = 1e-6

The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=SqueezeNet1_1_Weights.IMAGENET1K_V1`. You can also use `weights=SqueezeNet1_1_Weights.DEFAULT` to get the most up-to-date weights.


In [15]:
optimizer_g = torch.optim.Adam(params=autoencoder.parameters(), lr=1e-4)
optimizer_d = torch.optim.Adam(params=discriminator.parameters(), lr=1e-4)

### Train model

In [16]:
# Memory optimization settings
torch.backends.cudnn.benchmark = True  # Optimize for consistent input sizes
torch.backends.cuda.matmul.allow_tf32 = True  # Use TF32 for faster training
torch.backends.cudnn.allow_tf32 = True

# Enable gradient checkpointing to save memory
if hasattr(autoencoder, 'gradient_checkpointing'):
    autoencoder.gradient_checkpointing = True
if hasattr(discriminator, 'gradient_checkpointing'):
    discriminator.gradient_checkpointing = True

# Clear cache before training
torch.cuda.empty_cache()
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"GPU memory cached: {torch.cuda.memory_reserved()/1024**3:.2f} GB")

GPU memory allocated: 0.02 GB
GPU memory cached: 0.03 GB


### Visualise reconstructions

In [ ]:
autoencoder.load_state_dict(torch.load(os.path.join(ROOT_DIR+"AnoDiffExperiments/best_models/experiment_4", "exp_4_4_autoencoder_autoencoder_best.pth"), map_location=DEVICE_TYPE))
autoencoder.eval()


In [ ]:
autoencoder.eval()
val_recon_epoch_loss = 0
for step, batch in enumerate(test_reconstruction_loader):
    with torch.cuda.amp.autocast():
        print(f"Validation step {step} of epoch {epoch}")
        images = batch.to(device)  
        recons_loss = intensity_loss(
                    reconstruction.float(), images.float()
                ) + perceptual_weight * loss_perceptual(reconstruction.float(), images.float())
